# Pipeline Preditivo - Indústria 4.0
## Fase 1: Análise Exploratória (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carregando os dados (Certifique-se de que o arquivo CSV esteja na pasta /data)
df = pd.read_csv('data/base_de_dados.csv') # Ajuste o nome do arquivo exato aqui se for diferente

# Dimensões do dataset
print(f'Linhas: {df.shape[0]}, Colunas: {df.shape[1]}\n')

# Tipos de dados
print(df.dtypes)

# Resumo estatístico numérico
display(df.describe())


In [ ]:
plt.figure(figsize=(15, 4))

# 1. Histograma de uma variável preditora (Ajuste a variável alvo dependendo do CSV)
plt.subplot(1, 3, 1)
sns.histplot(df['velocidade_rotacao_rpm'], kde=True)
plt.title('Distribuição da Velocidade (RPM)')

# 2. Gráfico de barras da variável alvo (Desbalanceamento)
plt.subplot(1, 3, 2)
sns.countplot(x='Falha', data=df)
plt.title('Contagem da Variável Alvo (Falha)')

# 3. Mapa de calor de correlação de Pearson
plt.subplot(1, 3, 3)
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlação de Pearson')

plt.tight_layout()
plt.show()


**Análise da Fase 1:** Os gráficos demonstram a distribuição das variáveis numéricas. O gráfico de barras evidencia o desbalanceamento da variável alvo, com muito mais casos de funcionamento normal do que de falhas, o que exigirá reamostragem. O mapa de calor indica as variáveis com maior correlação linear.

## Fase 2: Limpeza e Tratamento de Dados (Data Prep)

In [ ]:
# Remover linhas duplicadas
df.drop_duplicates(inplace=True)

# Tratamento de nulos
# Imputação pela Mediana (menos sensível a outliers)
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)


**Justificativa (Nulos):** Optou-se pela imputação por Mediana em vez da Média para as variáveis contínuas, visto que ela é mais robusta e não sofre forte influência em caso de assimetria e presença de outliers extremos.

In [ ]:
# Boxplot para identificar outliers
plt.figure(figsize=(10, 5))
sns.boxplot(data=df.select_dtypes(include=np.number).drop(columns=['Falha'], errors='ignore'))
plt.xticks(rotation=45)
plt.title('Identificação de Outliers')
plt.show()


## Fase 3: Feature Engineering

In [ ]:
# Criando a coluna potência usando combinação matemática
if 'velocidade_rotacao_rpm' in df.columns and 'torque_nm' in df.columns:
    df['potencia'] = df['velocidade_rotacao_rpm'] * df['torque_nm']
    print('Feature de Potência criada com sucesso!')


## Fase 4: Divisão e Balanceamento dos Dados

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df.drop('Falha', axis=1)
y = df['Falha']

# Divisão 80/20 com estratificação
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Balanceamento apenas nos dados de treino usando SMOTE para evitar Data Leakage
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)


## Fase 5: Escalonamento de Variáveis (StandardScaler)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Aplicando fit_transform no treino e transform no teste (Apenas para o KNN)
X_train_knn = scaler.fit_transform(X_train_bal)
X_test_knn = scaler.transform(X_test)

# Os dados da Árvore de Decisão permanecem sem escalonamento: X_train_bal e X_test


**Justificativa (Escalonamento):** O KNN calcula distâncias euclidianas entre os pontos, sendo altamente sensível a variáveis em escalas diferentes (como rotação em milhares e torque em dezenas), por isso aplicou-se o StandardScaler. Já a Árvore de Decisão constrói regras baseadas na segregação (splits) de cada atributo independentemente, tornando o algoritmo imune à escala dos dados, dispensando esta etapa.

## Fase 6: Ajuste de Parâmetros e Combate ao Overfitting

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print('--- Avaliação KNN ---')
k_values = [3, 5, 7]
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train_bal)
    acc_train = accuracy_score(y_train_bal, knn.predict(X_train_knn))
    acc_test = accuracy_score(y_test, knn.predict(X_test_knn))
    print(f'K={k} -> Treino: {acc_train:.4f} | Teste: {acc_test:.4f}')

print('\n--- Avaliação Árvore de Decisão ---')
depths = [3, 5, None]
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_bal, y_train_bal)
    acc_train = accuracy_score(y_train_bal, dt.predict(X_train_bal))
    acc_test = accuracy_score(y_test, dt.predict(X_test))
    print(f'Max Depth={d} -> Treino: {acc_train:.4f} | Teste: {acc_test:.4f}')


**Análise de Overfitting:**
Na Árvore de Decisão, notamos que ao utilizar `max_depth=None`, geralmente ocorre *overfitting* (a acurácia de treino encosta nos 100%, mas a de teste sofre uma queda brusca). Uma profundidade restrita (3 ou 5) força a árvore a generalizar melhor e garante estabilidade no teste.
No modelo KNN, um K baixo (como 3) costuma gerar limites de decisão muito complexos (tendendo ao sobreajuste). Valores ligeiramente maiores estabilizam o modelo na base de teste.

## Fase 7: Avaliação da Acurácia e Veredito Final

In [ ]:
# Substitua 5 pelos melhores parâmetros identificados na etapa anterior
melhor_knn = KNeighborsClassifier(n_neighbors=5)
melhor_knn.fit(X_train_knn, y_train_bal)
acc_final_knn = accuracy_score(y_test, melhor_knn.predict(X_test_knn))

melhor_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
melhor_dt.fit(X_train_bal, y_train_bal)
acc_final_dt = accuracy_score(y_test, melhor_dt.predict(X_test))

print(f'Acurácia Final KNN: {acc_final_knn:.4f}')
print(f'Acurácia Final Árvore: {acc_final_dt:.4f}')


**Veredito:**
Ao avaliarmos os resultados na nossa massa independente de teste (20% separada inicialmente), o modelo [PREENCHER COM AQUI: KNN ou ÁRVORE] apresentou o desempenho superior. Por apresentar as maiores métricas e maior capacidade de generalização diante de novos dados sem o viés de overfitting, este é o algoritmo recomendado para compor o pipeline produtivo da empresa e atuar na identificação de quebras nos equipamentos.